# 🏢 VAT Risk Analysis Workshop — Exercise Notebook
### การวิเคราะห์ความเสี่ยงการยกเลิก VAT ด้วย PCA + K-Means + Machine Learning (สำหรับผู้เริ่มต้น)

---

📌 **วิธีใช้สมุดบันทึกนี้ / How to use this notebook:**
- อ่าน **Markdown cell** เพื่อทำความเข้าใจแต่ละขั้นตอน
- Read each **Markdown cell** to understand the step
- เติมโค้ดในช่อง **Code cell** ด้วยตัวเอง
- Fill in the **Code cell** with your own code

---

## 🎯 โจทย์ / Business Problem
> **"วิเคราะห์ข้อมูลผู้ประกอบการ VAT จำนวน 1.1 ล้านราย เพื่อค้นหาปัจจัยเชิงพื้นที่ที่มีผลต่อความเสี่ยงการยกเลิกทะเบียน VAT และสร้างโมเดลทำนาย"**
>
> **"Analyse 1.1M VAT business records to discover spatial risk factors and build a predictive model for VAT deregistration risk."**

---

## 📋 พจนานุกรมข้อมูล / Data Dictionary

| คอลัมน์ / Column | ประเภท / Type | ความหมาย / Meaning |
|---|---|---|
| เลขผู้เสียภาษีอากร | varchar | Tax ID |
| ตำบล | varchar | Sub-district / ตำบล |
| อำเภอ | varchar | District / อำเภอ |
| จังหวัด | varchar | Province / จังหวัด |
| รหัสไปรษณีย์ | varchar | Postal code |
| วันที่ได้รับอนุมัติ | varchar | VAT registration approval date (พ.ศ.) |

---
## 📦 ขั้นตอนที่ 0: นำเข้าไลบรารี / Step 0: Import Libraries

```python
import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
import matplotlib.pyplot as plt
import warnings

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import silhouette_score, classification_report

pio.renderers.default = 'notebook'
warnings.filterwarnings('ignore')
print("✅ Libraries imported!")
```

In [ ]:
# TODO: นำเข้าไลบรารีทั้งหมดตามตัวอย่างด้านบน
# Import all libraries as shown above


---
## 🗂️ ขั้นตอนที่ 1: โหลดข้อมูล / Step 1: Load Data

ข้อมูลผู้ประกอบการ VAT แบ่งเป็น 2 ไฟล์:  
VAT taxpayer data is split into 2 files:
- ไฟล์กรุงเทพฯ / Bangkok file: encoding = `tis-620`
- ไฟล์ต่างจังหวัด / Province file: encoding = `utf-8-sig`

```python
PATH_BKK  = r"E:\DGA_ALL\DGA_2\Revenue_Department\For_final\data\dga306.csv"
# หรือ / or path ของคุณ

cols_to_use = ['เลขผู้เสียภาษีอากร', 'จังหวัด', 'อำเภอ', 'ตำบล', 'รหัสไปรษณีย์ ', 'วันที่ได้รับอนุมัติ']

# โหลดข้อมูล (ถ้ามี 2 ไฟล์ ให้ concat, ถ้าไฟล์เดียวให้ใช้ pd.read_csv โดยตรง)
df = pd.read_csv(PATH_BKK, encoding='tis-620', encoding_errors='ignore',
                 usecols=cols_to_use, low_memory=False)
df.columns = df.columns.str.strip()
print(f"Shape: {df.shape}")
```

💡 **หมายเหตุ / Note:** ไฟล์ `dga306.csv` อาจมีข้อมูลทั้ง 2 ส่วนรวมกันแล้ว ให้โหลดด้วย encoding ที่เหมาะสม

In [ ]:
# TODO: โหลดไฟล์ข้อมูล VAT และแสดงขนาด
# Load VAT data file and print shape


---
## 🔭 ขั้นตอนที่ 2: สำรวจข้อมูลเบื้องต้น (EDA) / Step 2: Exploratory Data Analysis

### 2.1 ดูตัวอย่างข้อมูล / Preview data
```python
display(df.head())
```

In [ ]:
# TODO: แสดง 5 แถวแรก
# Display first 5 rows


### 2.2 ตรวจสอบโครงสร้าง / Check structure
```python
df.info()
```

In [ ]:
# TODO: แสดงโครงสร้างข้อมูล
# Show data structure


### 2.3 ตรวจสอบค่าว่าง / Check missing values
```python
print(df.isna().sum())
```

In [ ]:
# TODO: ตรวจสอบจำนวนค่าว่างในแต่ละคอลัมน์
# Check missing values per column


---
## 🧹 ขั้นตอนที่ 3: ทำความสะอาดข้อมูล / Step 3: Data Cleaning

### 3.1 แปลงวันที่จาก พ.ศ. → ค.ศ. / Convert Buddhist Era (BE) to Christian Era (CE)

⚠️ **ปัญหา / Problem:** วันที่ในข้อมูลเป็น พ.ศ. (เช่น `2559-09-22`) ต้องแปลงเป็น ค.ศ. โดยลบ 543  
⚠️ **Problem:** Dates are in Buddhist Era (e.g. `2559-09-22`). Convert to CE by subtracting 543.

```python
def clean_be_date(date_str):
    """แปลงวันที่ พ.ศ. → ค.ศ. / Convert BE date string to CE Timestamp."""
    if pd.isna(date_str) or not isinstance(date_str, str):
        return pd.NaT
    try:
        parts = date_str.split('-')
        if len(parts) == 3:
            year, month, day = int(parts[0]), int(parts[1]), int(parts[2])
            if year > 2400:  # ถ้าเป็น พ.ศ. ให้ลบ 543
                year -= 543
            return pd.Timestamp(year=year, month=month, day=day)
    except:
        pass
    return pd.NaT

df['registration_date'] = df['วันที่ได้รับอนุมัติ'].apply(clean_be_date)
df = df.dropna(subset=['registration_date'])  # ลบแถวที่แปลงไม่ได้
print(f"Rows after cleaning: {len(df):,}")
```

In [ ]:
# TODO: เขียนฟังก์ชัน clean_be_date และแปลงคอลัมน์วันที่
# Write clean_be_date function and convert date column


### 3.2 คำนวณอายุธุรกิจ / Calculate business age

```python
ref_date = pd.Timestamp('2026-07-01')
df['business_age_years'] = (ref_date - df['registration_date']).dt.days / 365.25
df = df[df['business_age_years'] >= 0]  # กรองข้อมูลผิดปกติ

# ทำความสะอาดช่องว่างส่วนเกิน
for col in ['จังหวัด', 'อำเภอ', 'ตำบล']:
    df[col] = df[col].astype(str).str.strip()

print(f"อายุธุรกิจเฉลี่ย / Average business age: {df['business_age_years'].mean():.1f} years")
```

In [ ]:
# TODO: คำนวณอายุธุรกิจและทำความสะอาดช่องว่าง
# Calculate business age and strip whitespace


---
## ⚙️ ขั้นตอนที่ 4: Feature Engineering — สกัดคุณลักษณะ

### 4.1 ความหนาแน่นธุรกิจตามรหัสไปรษณีย์ / Business Density by Postcode

นับจำนวนธุรกิจในแต่ละรหัสไปรษณีย์ เพื่อเป็น feature บ่งบอกการแข่งขัน  
Count the number of businesses per postcode as a competition indicator.

```python
postcode_counts = df['รหัสไปรษณีย์'].value_counts().to_dict()
df['postcode_business_density'] = df['รหัสไปรษณีย์'].map(postcode_counts)
print(df['postcode_business_density'].describe())
```

In [ ]:
# TODO: คำนวณ postcode_business_density
# Calculate postcode business density


### 4.2 สุ่มตัวอย่างข้อมูล / Sample data for efficient processing

ข้อมูลมี 1.1 ล้านแถว ซึ่งอาจช้า ให้สุ่มตัวอย่าง 100,000 แถวเพื่อเพิ่มความเร็ว  
Data has 1.1M rows which may be slow. Sample 100,000 rows for speed.

```python
df_sample = df.sample(n=min(100_000, len(df)), random_state=42).reset_index(drop=True)
print(f"Sample size: {len(df_sample):,}")
display(df_sample.head())
```

In [ ]:
# TODO: สุ่มตัวอย่าง 100,000 แถว
# Sample 100,000 rows


---
## 📉 ขั้นตอนที่ 5: PCA — ลดมิติข้อมูล / Step 5: PCA Dimensionality Reduction

### 5.1 Scale ข้อมูล / Standardize features

ก่อน PCA ต้อง Scale ข้อมูลให้อยู่ในสเกลเดียวกันด้วย `StandardScaler`  
Before PCA, use `StandardScaler` to put all features on the same scale.

```python
feature_cols = ['business_age_years', 'postcode_business_density']
X_raw = df_sample[feature_cols].dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
print(f"Shape after scaling: {X_scaled.shape}")
```

In [ ]:
# TODO: เลือก features และทำ StandardScaler
# Select features and apply StandardScaler


### 5.2 รัน PCA / Run PCA

```python
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f"Explained Variance Ratio: {pca.explained_variance_ratio_}")
print(f"Total Explained: {pca.explained_variance_ratio_.sum()*100:.1f}%")

plt.figure(figsize=(8, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.2, s=3)
plt.title('PCA 2D Projection')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.tight_layout()
plt.show()
```

In [ ]:
# TODO: รัน PCA และวาดกราฟ 2D
# Run PCA and plot 2D projection


---
## 🎯 ขั้นตอนที่ 6: K-Means Clustering / Step 6: K-Means Clustering

### 6.1 หาจำนวนกลุ่มที่เหมาะสม / Find optimal k

ทดลอง k = 2 ถึง 8 และวาด Elbow + Silhouette  
Try k = 2 to 8 and plot Elbow + Silhouette charts.

```python
wcss = []
sil  = []
k_range = range(2, 9)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    wcss.append(km.inertia_)
    sil.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(k_range, wcss, marker='o')
axes[0].set_title('Elbow Method')
axes[0].set_xlabel('k')
axes[0].set_ylabel('WCSS')

axes[1].plot(k_range, sil, marker='s', color='green')
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Score')
plt.tight_layout()
plt.show()
```

In [ ]:
# TODO: ทดลอง k=2 ถึง 8 และวาดกราฟ Elbow + Silhouette
# Try k=2 to 8 and plot both charts


### 6.2 ใส่ label กลุ่มและแสดงบน PCA / Assign labels and visualize on PCA

```python
BEST_K = 3  # <-- แก้ตามค่าที่เหมาะสม / change based on your analysis

km_final = KMeans(n_clusters=BEST_K, random_state=42, n_init=10)
cluster_labels = km_final.fit_predict(X_scaled)
df_sample_cluster = df_sample.loc[X_raw.index].copy()
df_sample_cluster['Cluster'] = cluster_labels

plt.figure(figsize=(9, 6))
for c in range(BEST_K):
    mask = cluster_labels == c
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], label=f'Cluster {c}', alpha=0.4, s=6)
plt.title(f'K-Means (k={BEST_K}) on PCA Space')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend(markerscale=3)
plt.tight_layout()
plt.show()

display(df_sample_cluster.groupby('Cluster')[['business_age_years', 'postcode_business_density']].mean().round(1))
```

In [ ]:
# TODO: รัน K-Means ด้วย k ที่เลือก และแสดงบน PCA
# Run K-Means with chosen k and visualize on PCA


---
## 🤖 ขั้นตอนที่ 7: Machine Learning — Random Forest Classifier

### 7.1 สร้าง Target Variable / Create target variable

สมมติว่า ธุรกิจที่มีอายุ < 3 ปี มี "ความเสี่ยงสูง" = 1  
Assume businesses with age < 3 years have "high risk" = 1.

```python
df_sample_cluster['high_risk'] = (df_sample_cluster['business_age_years'] < 3).astype(int)
print("Target distribution:")
print(df_sample_cluster['high_risk'].value_counts())
```

In [ ]:
# TODO: สร้าง target variable high_risk
# Create high_risk target variable


### 7.2 เทรนและประเมินโมเดล / Train and evaluate model

```python
X_ml = df_sample_cluster[['business_age_years', 'postcode_business_density']].dropna()
y_ml = df_sample_cluster.loc[X_ml.index, 'high_risk']

X_tr, X_te, y_tr, y_te = train_test_split(X_ml, y_ml, test_size=0.2, random_state=42, stratify=y_ml)

clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf.fit(X_tr, y_tr)
y_pred = clf.predict(X_te)

print(classification_report(y_te, y_pred, target_names=['Low Risk', 'High Risk']))
```

In [ ]:
# TODO: สร้าง Random Forest Classifier และแสดง classification report
# Build Random Forest Classifier and show classification report


### 7.3 Feature Importance / ความสำคัญของ Features

```python
importances = pd.Series(clf.feature_importances_, index=X_ml.columns).sort_values()

plt.figure(figsize=(6, 3))
importances.plot(kind='barh', color='steelblue', edgecolor='white')
plt.title('Feature Importance — Random Forest')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()
```

In [ ]:
# TODO: แสดง Feature Importance ด้วยกราฟ
# Plot Feature Importance


---
## 🏁 สรุป / Summary

ยินดีด้วย! คุณได้เรียนรู้ขั้นตอนต่อไปนี้แล้ว / Congratulations! You have completed:

| ขั้นตอน / Step | หัวข้อ / Topic |
|---|---|
| 0 | Import Libraries |
| 1 | Load VAT data (multi-encoding) |
| 2 | EDA (head, info, isna, describe) |
| 3 | Data Cleaning (BE→CE date conversion, business age) |
| 4 | Feature Engineering (business density) |
| 5 | PCA — dimensionality reduction + 2D visualization |
| 6 | K-Means Clustering (Elbow + Silhouette) |
| 7 | Random Forest Classifier + Feature Importance |

📂 **ดูเฉลย:** เปิดไฟล์ `vat_teaching_solution.ipynb`  
📂 **See solution:** Open `vat_teaching_solution.ipynb`